# Berry–Esseen: n2 i n3 — pełne obliczenia CPU

Wybierz **Uruchom wszystko**. Notebook sprawdza całe dostarczone certyfikaty n2 i n3,
kompiluje jądro C++ ze źródeł i zapisuje postęp.
Zgodność oryginalnej ochrony Windows i nowego wykonania Linux jest raportowana osobno.
Sam ten wynik nie zamyka C4 ani całego twierdzenia.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
CODE_COMMIT = "3a9935abd7337f5e0ce5b5e798a9c88a064eb786"
TASK = "n2-n3"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Pobieram przypięty kod:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Niezgodny commit źródeł")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
print(json.dumps(snapshot(), indent=2), flush=True)
print("Kod i środowisko odczytane. To jeszcze nie akceptacja matematyczna.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logi i checkpointy:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Sprawdzam nadzorcę na rzeczywistym Linuksie. To lekkie testy, bez dowodu.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Test nadzorcy nie przeszedł; obliczenia nie wystartują. Log: " + str(SELFTEST))
print("Test nadzorcy zakończony. Rozpoczynam właściwe zadanie.", flush=True)


In [ ]:
import shutil
if shutil.which("g++") is None or not Path("/usr/include/boost/multiprecision/cpp_int.hpp").exists():
    print("Instaluję kompilator i nagłówki Boost z repozytorium systemowego Colaba.", flush=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "g++", "libboost-dev"], check=True)
print("Kompilator:", subprocess.check_output(["g++", "--version"], text=True).splitlines()[0], flush=True)
from be_replay.small_indices import run_colab
RESULT = run_colab(durable_root=DURABLE)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Stan:", RESULT["status"]["state"], flush=True)
print("Globalny dowód nie jest oznaczony jako zamknięty.", flush=True)
print("ZIP na Dysku:", RESULT["bundle"], flush=True)
display(FileLink(RESULT["bundle"]))
try:
    files.download(RESULT["bundle"])
except Exception as error:
    print("Pobierz ZIP z podanej ścieżki na Dysku:", error, flush=True)
